# Capstone — mirrors the deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-9862873/FlyRank-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook is the runnable twin of the deployed paper. Every number the paper prints is either recomputed here from the dataset (env-stable facts) or read from the committed receipts (`work/outputs/playbook_metrics.json`), so the paper's claims can be checked without retraining anything.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `writing-research-papers` + `deploying-static-pages` for this task.

## 1. Question

*The research question and the decision it supports.*

Content teams manage tens of thousands of pages and cannot review them all by hand. The question: **can observed search and behavior signals rank pages by their odds of declining, so a human works the top of a queue instead of guessing?** The decision this supports is weekly refresh triage — which page to open first — not an automated edit. The paper's framing stays decision-support throughout.

In [1]:
import pandas as pd, numpy as np, json
from pathlib import Path

REPO = Path('../..')
df = pd.read_csv(REPO / 'data' / 'raw' / 'content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f'Question under study: rank {len(df):,} pages by odds of decline for weekly triage.')
print(f'Base rate (declining): {df["is_declining_label"].mean():.1%} — almost a coin flip to begin with.')

Question under study: rank 30,000 pages by odds of decline for weekly triage.
Base rate (declining): 54.2% — almost a coin flip to begin with.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The anonymized starter release of the FlyRank ML internship dataset: 30,000 content pages, one row per page, 32 clients, 45 shipped columns. Windows are relative (90-day aggregates; the label compares the last 30 days to the previous 30). Absolute dates were removed in anonymization, so no calendar window can be named. Public-safe throughout: anonymized IDs only — no client names, URLs, or queries appear anywhere in this work.

In [2]:
print(f'Rows (pages): {len(df):,}   Clients: {df["client_id"].nunique()}   Columns shipped: {df.shape[1]}')
print()
print('Content archetypes:')
print(df['content_type'].value_counts().to_string())
print()
print(f'word_count missing: {df["word_count"].isna().sum():,} ({df["word_count"].isna().mean():.1%}) — feedly-type articles carry no keyword data.')
print(f'avg_position == 0 (dictionary: NO DATA, not rank zero): {(df["avg_position"] == 0).sum():,} pages → modeled via a has_position flag, never as a literal rank.')
print()
print('Excluded from features on purpose: trend_direction, trend_pct (label-derived),')
print('provider_used / model_used (production metadata, not page signals),')

Rows (pages): 30,000   Clients: 32   Columns shipped: 45

Content archetypes:
content_type
keyword article       27207
feedly article         2096
comparison article      697

word_count missing: 7,699 (25.7%) — feedly-type articles carry no keyword data.
avg_position == 0 (dictionary: NO DATA, not rank zero): 1,205 pages → modeled via a has_position flag, never as a literal rank.

Excluded from features on purpose: trend_direction, trend_pct (label-derived),
provider_used / model_used (production metadata, not page signals),


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

- **Label:** `trend_direction == 'down'` → binary `is_declining_label` (base rate 54.2%).
- **Features (12):** ten 90-day aggregates plus `has_word_count` and `has_position`. Label-derived columns are never features.
- **Baseline:** the industry-style rule from ML-07 — stale ≥180 days AND ≥500 impressions (`stale_visible`), scored by impressions.
- **Models:** logistic regression (scaled) and random forest (200 trees, depth 10, seed 42).
- **Validation:** client-grouped 80/20 split (seed 42) — the test set is 6,163 pages from 7 clients the model has never seen. A random split is also scored, only to measure how much the grouped number is being dragged down by memorization.
- **Leakage checks (ML-09):** no label columns or target encodings in features; feature window overlaps the label window by 30 days (disclosed, LOW); the random-vs-grouped AUC gap quantifies client memorization.

In [3]:
feature_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d',
                'avg_position', 'ctr', 'engagement_rate', 'content_age_days',
                'days_since_last_update', 'word_count']
print('Features:', len(feature_cols), 'aggregates + has_word_count + has_position = 12 total')
assert 'trend_direction' not in feature_cols and 'trend_pct' not in feature_cols

receipts = json.load(open(REPO / 'work' / 'outputs' / 'playbook_metrics.json', encoding='utf-8'))
g = receipts['metrics']['grouped']
print(f"Grouped test: {g['n_test']:,} pages from {g['n_test_clients']} unseen clients")
print('All seeds fixed at 42; split, metrics, and queue are deterministic given the release.')

Features: 10 aggregates + has_word_count + has_position = 12 total
Grouped test: 6,163 pages from 7 unseen clients
All seeds fixed at 42; split, metrics, and queue are deterministic given the release.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The model beats the rule where it matters — the top of the queue — but its overall discrimination on unseen clients is weak. The random-split AUC that looked respectable (0.757 for the RF) collapses to 0.603 when whole clients are held out: a 0.154 memorization gap, which is why every headline number here is the grouped one.

In [4]:
rows = [
    ['Baseline (rule)',  g['baseline_p10'], g['baseline_p20'], g['baseline_p50'], None, None],
    ['Logistic Regression', g['lr_p10'], g['lr_p20'], g['lr_p50'], round(g['lr_f1'], 3), round(g['lr_auc'], 3)],
    ['Random Forest',    g['rf_p10'], g['rf_p20'], g['rf_p50'], round(g['rf_f1'], 3), round(g['rf_auc'], 3)],
]
table = pd.DataFrame(rows, columns=['method', 'P@10', 'P@20', 'P@50', 'F1', 'AUC'])
print('Client-grouped test set (the only honest numbers):')
print(table.to_string(index=False))
print()
r = receipts['metrics']['random']
print(f"Random split for contrast: LR AUC {r['lr_auc']:.3f}, RF AUC {r['rf_auc']:.3f}")
print(f"Memorization gap (random - grouped): LR {receipts['metrics']['memorization_gap_auc']['lr']:.3f}, RF {receipts['metrics']['memorization_gap_auc']['rf']:.3f}")
print()
print('Footnote on the baseline: its score is a step function with thousands of ties;')
print('top-k selection among ties is numpy-version dependent (0.5-0.8 observed across')
print('environments). Values above match the canonical ML-08 receipts (venv, py3.10).')

Client-grouped test set (the only honest numbers):
             method  P@10  P@20  P@50    F1   AUC
    Baseline (rule)   0.5  0.45  0.62   NaN   NaN
Logistic Regression   0.6  0.70  0.68 0.644 0.573
      Random Forest   0.7  0.70  0.60 0.621 0.603

Random split for contrast: LR AUC 0.655, RF AUC 0.757
Memorization gap (random - grouped): LR 0.082, RF 0.154

Footnote on the baseline: its score is a step function with thousands of ties;
top-k selection among ties is numpy-version dependent (0.5-0.8 observed across
environments). Values above match the canonical ML-08 receipts (venv, py3.10).


## 5. Limitations

*What this work cannot claim.*

1. Grouped AUC 0.603 — directional ranking only, barely above chance.
2. Client memorization: features leak client identity; the gap (0.154) is measured, not hidden.
3. Feature window overlaps the label window by 30 days (disclosed).
4. Decay is non-monotonic: the 91-180d bucket declines most (61.1%); the 181-365d (n=169) and 365d+ (n=5) buckets are too thin to quote.
5. Production panel, no interventions — nothing here proves refreshing a page recovers traffic.
6. Single snapshot — no time-based holdout is possible in this release.
7. Baseline P@k is tie-breaking unstable across environments (see footnote above).

In [5]:
decay = receipts['decay_buckets']
print('Decline rate by staleness (with n — thin buckets are labeled as thin):')
for b in decay:
    print(f"  {b['staleness_bucket']:>9}: {b['decline_rate']:.1%}  (n={b['pages']:,})")
print()
print('Claim language used on the paper: observed, measured, directional, decision-support.')

Decline rate by staleness (with n — thin buckets are labeled as thin):
      0-90d: 51.2%  (n=20,655)
    91-180d: 61.1%  (n=9,171)
   181-365d: 46.7%  (n=169)
      365d+: 60.0%  (n=5)

Claim language used on the paper: observed, measured, directional, decision-support.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

A weekly queue of all 30,000 pages ranked by the RF's decline probability, with a human-readable reason code per row. The model sets priority; the reason code names the kind of look a page needs: `insufficient_signal` → NO_ACTION (7,994), `stale_visible` → REFRESH (17), `high_traffic_weak_position` → REOPTIMIZE (3,518), everything else → REVIEW (18,471). Humans check CMS liveness, external causes, and brand constraints before any edit; publishing, deleting, position-zero rows, and thin staleness buckets are never automated.

In [6]:
q = receipts['queue']
print(f"Queue size: {q['rows']:,} pages")
for action, n in sorted(q['actions'].items(), key=lambda kv: -kv[1]):
    print(f'  {action:>10}: {n:,}')
print()
print('Monitoring triggers (from the playbook):')
for k, v in receipts['monitoring'].items():
    print(f'  {k}: {v}')

Queue size: 30,000 pages
      REVIEW: 18,471
   NO_ACTION: 7,994
  REOPTIMIZE: 3,518
     REFRESH: 17

Monitoring triggers (from the playbook):
  checked_against: starter dataset v30k, 2026 snapshot
  base_rate_current: 0.5421
  base_rate_retrain_trigger: shift > 0.05 absolute on a fresh labeled batch
  p20_floor: 0.5
  impressions_median_current: 731.0
  impressions_drift_trigger: median moves > 30% from current
  new_client_rule: clients with < 90 days history are excluded until a full window exists
  retrain_cadence: when any trigger fires, or every 6 months whichever comes first


## 7. Artifacts the paper embeds

*Charts and receipts the deployed page shows — all committed, all regenerable.*

In [7]:
for p in sorted((REPO / 'work' / 'figures').glob('*.png')):
    print(f'  work/figures/{p.name}  ({p.stat().st_size:,} bytes)')
for name in ['playbook_metrics.json', 'baseline_action_score.csv', 'content_action_queue.csv']:
    p = REPO / 'work' / 'outputs' / name
    if p.exists():
        print(f'  work/outputs/{name}  ({p.stat().st_size:,} bytes)' + ('' if name.endswith('.json') else '  [regenerable, excluded from git by leak-guard]'))
print()
print('Paper: index.html at repo root → https://abdullah-9862873.github.io/FlyRank-Internship/')

  work/figures/fig_decline_by_staleness.png  (34,648 bytes)
  work/figures/fig_memorization_gap.png  (29,635 bytes)
  work/figures/fig_precision_at_k.png  (31,865 bytes)
  work/figures/fig_queue_actions.png  (33,186 bytes)
  work/outputs/playbook_metrics.json  (2,392 bytes)
  work/outputs/baseline_action_score.csv  (2,553,637 bytes)  [regenerable, excluded from git by leak-guard]
  work/outputs/content_action_queue.csv  (3,995,659 bytes)  [regenerable, excluded from git by leak-guard]

Paper: index.html at repo root → https://abdullah-9862873.github.io/FlyRank-Internship/


## 8. Five-minute demo outline (Week-8 showcase)

*Question, method, one chart, one honest result, one recommendation — timed to five minutes.*

**0:00–0:40 — The question.** FlyRank's oldest customer question: tens of thousands of indexed pages, most quietly decaying, and no team that can review them all. Can observed search and behavior signals rank pages by their odds of declining, so a human works the top of a queue instead of guessing? The decision supported is weekly triage order — not an autopilot.

**0:40–1:30 — The data.** 30,000 real production pages, 32 clients, anonymized (the FlyRank ML internship dataset). Label = declining over the last 30 days vs the previous 30. Base rate 54.2% — barely better than a coin flip is the bar.

**1:30–2:45 — The method.** A two-condition freshness rule (stale ≥180d AND ≥500 impressions) as the baseline to beat. Then logistic regression and a random forest on twelve observed features — no label-derived columns. Validation is split *by client* (test = 6,163 pages from 7 clients never seen in training), plus a leakage audit; a random split is scored only to measure memorization.

**2:45–3:45 — One chart.** `fig_precision_at_k.png`. On unseen clients, the forest puts declining pages in the top 10 at 0.70 versus the baseline's 0.50. The model earns priority ordering at the top of the queue — not blanket trust.

**3:45–4:30 — One honest result.** Overall grouped AUC is only 0.603. The random split said 0.757; holding out whole clients drops it by 0.154 — client memorization, measured and shown as its own chart (`fig_memorization_gap.png`). That gap is the true ceiling, and it's in the paper, not buried.

**4:30–5:00 — One recommendation.** Ship it as decision-support: a 30,000-page queue with reason codes (REFRESH 17, REOPTIMIZE 3,518, REVIEW 18,471, NO_ACTION 7,994), a mandatory human-review protocol, and monitoring triggers (base rate ±5pp, P@20 floor 0.5). Never auto-publish, never trust a lone AUC.

**Close (10 seconds).** The most valuable number in this project is the one that made the model look worse.

## 9. Two shareable cuts

*The same work, retold for two audiences — ready to paste as-is.*

### Cut 1 — social post (methodology story, LinkedIn-length)

I kept seeing ML write-ups report a random-split AUC and move on. On my FlyRank capstone — predicting which of 30,000 production content pages are declining — the random split said my random forest hit 0.757 AUC.

Then I grouped the split by client, so the model couldn't quietly learn "this traffic signature = this client." The AUC fell to 0.603. That 0.154 collapse isn't a bug in my report — it's the result. It's a chart in the paper.

The baseline was a two-condition freshness rule anyone could write on a whiteboard (stale ≥180 days, still visible). My forest beat it where a queue actually lives — precision@10 of 0.70 vs 0.50 on clients it had never seen — but only modestly, and I say so.

Takeaway I'll carry into every future project: if a known entity (client, user, site) shows up in your features, group your split by that entity or your metric is measuring memorization, not generalization.

Paper + notebooks: https://abdullah-9862873.github.io/FlyRank-Internship/

### Cut 2 — employer-facing 3-sentencer

**What I built:** a ranked content-refresh queue (30,000 pages) with human-readable reason codes, a mandatory human-review protocol, and monitoring triggers — decision-support for weekly content triage, shipped as a deployed research page with reproducible notebooks.

**On what data:** the anonymized FlyRank production search dataset — 30,000 real pages across 32 clients — evaluated with client-grouped validation (test set = 7 unseen clients) and a full leakage audit; a rule baseline (stale ≥180d and visible) was the comparison to beat.

**What it showed:** the random forest beat the baseline at the top of the queue (precision@10 0.70 vs 0.50 on unseen clients) but discriminates only weakly overall (AUC 0.603), and a measured 0.154 random-vs-grouped memorization gap set the honest ceiling — so the system ships as decision-support with human review, never as an autopilot.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] This notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Committed under `work/notebooks/` — repo URL submitted on the card
- [x] Deployed paper has all 9 sections — Abstract at the top, Acknowledgments & data credit (flyrank.ai link) at the bottom
- [x] ML-12 done in the closing cells: 5-minute demo outline + social-post cut + employer 3-sentencer